[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-13.ipynb)

# 13주차: 논문 방법의 작은 CPU 재현

- 관찰 전용: 모든 셀을 수정 없이 실행, 학생 알고리즘 구현 없음
- Self-Refine §2의 생성→피드백→수정 한 회 구조를 제공 코드로 실행
- 실제 공개 모델 생성, 교수 작성 점수나 모의 모델 호출 없음
- Qwen3-0.6B-Base CPU float32, 두 연산 스레드, greedy, 호출당 최대 24 생성 토큰
- 모델 가중치 다운로드와 인터넷 필요. 모델 로딩·설치는 비교 지연에서 제외하고 따로 기록
- 원 논문의 모델·과제·few-shot 프롬프트·반복 중단 조건과 다름. 원 성능 수치·학습·SAIGE 재현 아님
- Base 모델은 지시 학습 모델과 다름. 형식 실패도 결과로 유지
- [Self-Refine (2023), §2·§4·§6](https://arxiv.org/html/2303.17651v2)
- [Rethinking Multi-Agent Collaboration (2026), §3·§5](https://arxiv.org/html/2609.19759v1): 읽기용, 해당 시스템 구현 없음
- [Qwen3 Technical Report (2025)](https://arxiv.org/abs/2505.09388), [모델 카드](https://huggingface.co/Qwen/Qwen3-0.6B-Base)


In [1]:
# 실행 환경 보완: 없는 라이브러리만 지정 버전으로 설치, 기존 환경은 업그레이드하지 않음.
import importlib.metadata
import subprocess, sys
requirements = {"torch": "2.8.0", "transformers": "4.57.6"}
missing = []
for name, version in requirements.items():
    try:
        importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        missing.append(f"{name}=={version}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()
for name in requirements:
    print(name, importlib.metadata.version(name))


torch 2.8.0+cpu
transformers 4.57.6


## 1. 고정판의 공개 근거 수집

- 공개 행정시 사실을 실제 문단과 대조. 원문 전체 파일 저장 없음
- 고정판과 제목 불일치·네트워크 오류는 명시적으로 실패, 가짜 fallback 없음
- CC BY-SA 4.0. 문서 기여자와 이용 조건은 원문·이력·이용약관 참조


In [2]:
import hashlib
import html
import json
import math
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (https://deepnlp-2026.halla.ai/)"})
for attempt in range(3):
    try:
        with urllib.request.urlopen(request, timeout=40) as response:
            raw_response = response.read()
        parsed = json.loads(raw_response)["parse"]
        break
    except (urllib.error.URLError, TimeoutError) as error:
        if attempt == 2:
            raise RuntimeError("공개 원문 수집 실패. 연결·출처 확인 후 다시 실행. 대체 문서 없음") from error
        time.sleep(2 ** attempt)
assert parsed["revid"] == OLDID, "고정판 번호 불일치"
assert parsed["title"] == "제주특별자치도", "원문 제목 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    fragment = re.sub(r"<[^>]+>", " ", fragment)
    return re.sub(r"\s+", " ", html.unescape(fragment)).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
evidence = [p for p in paragraphs if all(term in p for term in ("행정시", "제주시", "서귀포시"))]
assert evidence, "두 행정시의 원문 문단을 확인하지 못함. 자동 대체 금지"
source_metadata = {
    "url": SOURCE, "title": parsed["title"], "revision": parsed["revid"],
    "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    "response_sha256": hashlib.sha256(raw_response).hexdigest(),
    "attribution": "한국어 위키백과 제주특별자치도 문서와 해당 판의 기여자; 저자·이용 조건은 문서 이력 참조",
}
print("출처 메타데이터:", source_metadata)
print("실제 공개 원문 근거:", evidence[0])
source_metadata['license'] = 'CC BY-SA 4.0'
source_metadata['history_url'] = 'https://ko.wikipedia.org/w/index.php?title=제주특별자치도&action=history'
source_metadata['terms_url'] = 'https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use'
context = evidence[0]
print('근거 문자 수:', len(context))


출처 메타데이터: {'url': 'https://ko.wikipedia.org/w/index.php?oldid=42464087', 'title': '제주특별자치도', 'revision': 42464087, 'fetched_at_utc': '2026-11-28T07:54:34.929953+00:00', 'response_sha256': '51bbbe23bfae3fdfef15fb0984201fc6d872cd55c6894731e7e571329475575b', 'attribution': '한국어 위키백과 제주특별자치도 문서와 해당 판의 기여자; 저자·이용 조건은 문서 이력 참조'}
실제 공개 원문 근거: 조선 시대에는 전라도 에 속했으나, 1946년 8월 전라남도 ,에서 분리되어 제주도(濟州道)로 승격되었다. 2006년 7월 1일부터 특별자치도 로 승격되어 자치경찰의 실시, 교육자치권의 확대, 일부 중앙권한의 이양, 자치입법권, 자치재정권의 부여 등 지역적 자치권이 고도화되었다. 하위에 기초지방자치단체 를 두지 않은 단층제 광역자치단체이며, 남북으로 두 개의 행정시 인 제주시 와 서귀포시 가 설치되어 있다.
근거 문자 수: 229


## 2. 모델과 실행 조건 고정

- 가중치 revision 고정. CPU, float32, eval, gradient 없음
- 실제 파라미터 수·가중치 바이트·라이브러리·CPU·모델 로드 시간을 출력
- 가중치 바이트는 RSS가 아님. KV 캐시·토크나이저·중간 텐서·운영체제 메모리 별도


In [3]:
import platform
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = 'Qwen/Qwen3-0.6B-Base'
REVISION = 'da87bfb608c14b7cf20ba1ce41287e8de496c0cd'
torch.set_num_threads(2)
torch.manual_seed(0)
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.float32).to('cpu').eval()
load_sec = time.perf_counter() - t0
parameter_count = sum(p.numel() for p in model.parameters())
weight_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
conditions = dict(model=MODEL, revision=REVISION, device='cpu', dtype='float32', threads=2,
                 max_new_tokens=24, do_sample=False, use_model_defaults=False, seed=0, torch=torch.__version__,
                 transformers=transformers.__version__, python=platform.python_version(),
                 machine=platform.machine(), load_sec=load_sec,
                 parameter_count=parameter_count, weight_bytes=weight_bytes)
print(json.dumps(conditions, ensure_ascii=False, indent=2))


tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`torch_dtype` is deprecated! Use `dtype` instead!


{
  "model": "Qwen/Qwen3-0.6B-Base",
  "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "device": "cpu",
  "dtype": "float32",
  "threads": 2,
  "max_new_tokens": 24,
  "do_sample": false,
  "use_model_defaults": false,
  "seed": 0,
  "torch": "2.8.0+cpu",
  "transformers": "4.57.6",
  "python": "3.10.0",
  "machine": "AMD64",
  "load_sec": 4.298014799999919,
  "parameter_count": 596049920,
  "weight_bytes": 2384199680
}


## 3. 제공된 생성·검토·수정 함수

- 프롬프트 세 종류가 같은 모델 가중치 공유. 서로 다른 독립 모델·학습 에이전트가 아님
- 예열 한 호출의 시간과 토큰은 비교 합계에서 제외
- 호출 시간: 토큰화 이후 generate 구간 wall time. 프리필+디코딩 포함, 순수 디코딩 시간 아님
- 로그에는 프롬프트 hash·출력·토큰·시간만 저장. 공개 원문·전체 프롬프트 저장 없음


In [4]:
def ask(prompt, stage):
    batch = tok(prompt, return_tensors='pt')
    started = time.perf_counter()
    with torch.inference_mode():
        # 호환성 보완: 명시한 greedy 설정에 모델 기본 생성값을 다시 상속하지 않음.
        out = model.generate(**batch, max_new_tokens=24, do_sample=False, use_model_defaults=False,
                             pad_token_id=tok.eos_token_id, use_cache=True)
    elapsed = time.perf_counter() - started
    new_ids = out[0, batch.input_ids.shape[1]:]
    return dict(stage=stage, text=tok.decode(new_ids, skip_special_tokens=True).strip(),
                input_tokens=int(batch.input_ids.shape[1]), output_tokens=int(new_ids.numel()),
                generate_sec=elapsed, prompt_sha256=hashlib.sha256(prompt.encode()).hexdigest())

warmup = ask('제주특별자치도의 행정시는', 'warmup')
print('예열(비교 제외):', warmup)
cases = [
    dict(id='cities', question='제주특별자치도의 두 행정시 이름은?', expected=['제주시','서귀포시']),
    dict(id='count', question='제주특별자치도의 행정시 수는?', expected=['2']),
    dict(id='other', question='제주특별자치도에서 제주시 외의 행정시 이름은?', expected=['서귀포시']),
]
def base_prompt(case):
    return f"문서: {context}\n질문: {case['question']}\n문서에 있는 정답만 한 줄로 쓰시오. 수는 아라비아 숫자로 쓰시오.\n답:"

def match(text, case):
    # 첫 줄의 답 형식만 평가. 한국어 별칭·설명·부정문은 자동으로 정답 인정하지 않음.
    first = text.splitlines()[0].strip() if text.strip() else ''
    normalized = re.sub(r'[\s,·/;:.]', '', first)
    if case['id'] == 'cities':
        return normalized in ('제주시서귀포시', '서귀포시제주시')
    return normalized == case['expected'][0]

assert match('제주시, 서귀포시', cases[0])
assert not match('제주시와 서귀포시가 아닙니다', cases[0])
assert not match('서귀포시는 아닙니다', cases[2])


예열(비교 제외): {'stage': 'warmup', 'text': '2019년 1월 1일부터 2023년 12월', 'input_tokens': 11, 'output_tokens': 24, 'generate_sec': 3.210078799999792, 'prompt_sha256': '26656c8dfabd5243dd96ec5ea523ecfa6d671c5d2b06de3e6b14660cbb848026'}


## 4. 동일 초안에서 한 번의 Self-Refine 실행

- 먼저 direct 초안 생성, 초안과 근거로 피드백 생성, 근거·초안·피드백으로 수정 생성
- direct와 refine 구성은 같은 초안을 공유. refine 비교 비용에는 초안 호출을 포함
- 총 비교용 실행은 9호출(3질문×3단계). 두 구성을 각각 운영하면 질문당 1호출 대 3호출
- 문맥과 이전 출력이 늘어난 효과, 프롬프트 형식, 모델 능력이 함께 바뀌므로 독립 에이전트의 인과 효과 판정 불가


In [5]:
runs = []
for case in cases:
    initial = ask(base_prompt(case), 'initial')
    feedback = ask(f"문서: {context}\n질문: {case['question']}\n초안: {initial['text']}\n문서와 초안을 대조하고 수정할 점 한 가지만 짧게 쓰시오.\n검토:", 'feedback')
    refined = ask(f"문서: {context}\n질문: {case['question']}\n초안: {initial['text']}\n검토: {feedback['text']}\n검토를 참고해 정답만 한 줄로 다시 쓰시오. 수는 아라비아 숫자로 쓰시오.\n답:", 'refined')
    rows = [initial, feedback, refined]
    direct = dict(calls=1, input_tokens=initial['input_tokens'], output_tokens=initial['output_tokens'],
                  generate_sec=initial['generate_sec'], format_match=match(initial['text'],case))
    refine = dict(calls=3, input_tokens=sum(r['input_tokens'] for r in rows),
                  output_tokens=sum(r['output_tokens'] for r in rows),
                  generate_sec=sum(r['generate_sec'] for r in rows), format_match=match(refined['text'],case))
    row = dict(case_id=case['id'], question=case['question'], expected=case['expected'],
               stages=rows, direct=direct, refine=refine)
    runs.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2))
assert len(runs) == 3 and all(len(r['stages'])==3 for r in runs)


{
  "case_id": "cities",
  "question": "제주특별자치도의 두 행정시 이름은?",
  "expected": [
    "제주시",
    "서귀포시"
  ],
  "stages": [
    {
      "stage": "initial",
      "text": "제주특별자치도의 두 행정시 이름은 제주시와 서귀포시이다.",
      "input_tokens": 226,
      "output_tokens": 23,
      "generate_sec": 4.575529299999971,
      "prompt_sha256": "85fdcec519b12787c90fb3b282451aa2232c300a7366b8b26930bf0eee595f11"
    },
    {
      "stage": "feedback",
      "text": "제주특별자치도의 두 행정시 이름은 제주시와 서귀포시이다.\n문서",
      "input_tokens": 245,
      "output_tokens": 24,
      "generate_sec": 4.4855175000000145,
      "prompt_sha256": "6c85d1315316c48615703477287ef83e925ccd1324cb21ce932c91680c6534a4"
    },
    {
      "stage": "refined",
      "text": "제주특별자치도의 두 행정시 이름은 제주시와 서귀포시이다.\n문서",
      "input_tokens": 281,
      "output_tokens": 24,
      "generate_sec": 4.780497799999921,
      "prompt_sha256": "f38a2be2542b56ff0dbca588a6866f8562886926d751f040bc96963aa3c4905d"
    }
  ],
  "direct": {
    "calls": 1,
    "input_tokens"

{
  "case_id": "count",
  "question": "제주특별자치도의 행정시 수는?",
  "expected": [
    "2"
  ],
  "stages": [
    {
      "stage": "initial",
      "text": "2",
      "input_tokens": 225,
      "output_tokens": 3,
      "generate_sec": 1.633210100000042,
      "prompt_sha256": "6608aa1bc592e34040241c075c7053baa23633a10fda76cee198d2582d94722e"
    },
    {
      "stage": "feedback",
      "text": "제주특별자치도의 행정시 수는 2개인 제주시와 서귀포시가",
      "input_tokens": 225,
      "output_tokens": 24,
      "generate_sec": 4.3751371000000745,
      "prompt_sha256": "1b91c70b68b5ff629d377c17e12007b3c8ab8e879658cfdbad51fed41d86b16a"
    },
    {
      "stage": "refined",
      "text": "제주특별자치도의 행정시 수는 2개인 제주시와 서귀포시가",
      "input_tokens": 261,
      "output_tokens": 24,
      "generate_sec": 4.450753999999961,
      "prompt_sha256": "c9936e9d2c74b8b7e214ba0b260fde5fc2a884eee0c4545fb0221863b67c5f28"
    }
  ],
  "direct": {
    "calls": 1,
    "input_tokens": 225,
    "output_tokens": 3,
    "generate_sec": 1.633210

{
  "case_id": "other",
  "question": "제주특별자치도에서 제주시 외의 행정시 이름은?",
  "expected": [
    "서귀포시"
  ],
  "stages": [
    {
      "stage": "initial",
      "text": "제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제",
      "input_tokens": 230,
      "output_tokens": 24,
      "generate_sec": 4.466800400000011,
      "prompt_sha256": "adcfde54633a9149cd819c06b781b4f29923f30ed1578aa5249629256ca1d0a8"
    },
    {
      "stage": "feedback",
      "text": "제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제",
      "input_tokens": 252,
      "output_tokens": 24,
      "generate_sec": 4.686221499999874,
      "prompt_sha256": "0b5cbca507d358ffedc666897eb091a7e98a2c4b463d97e8dfd7a8b6f5a34abc"
    },
    {
      "stage": "refined",
      "text": "제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제",
      "input_tokens": 288,
      "output_tokens": 24,
      "generate_sec": 4.790374799999881,
      "prompt_sha256": "54fda115ac09cea719d5308ff660655d48b907c8918863826fca96025d52b9a3"
    }
  ],
  "direct": {
    "calls": 1,
    "input_tokens": 230,

## 5. 구성 비교표와 오답 전이

- 점수는 지정 첫 줄 형식 일치. 의미적 사실 정확도·faithfulness·벤치마크 성능과 다름
- 정답을 놓친 형식 오류는 출력과 함께 수동 검토. 규칙을 결과 확인 후 유리하게 바꾸지 않음
- 비용은 토큰·호출·시간의 대리 지표. 현금 비용·전력·RSS·서버 동시 처리량은 미측정
- 3개 질문은 통계적 우위 주장에 부족. 같은 조건 repeated timing·다른 문서·더 많은 질문은 후속 연구


In [6]:
comparison = []
for method in ('direct','refine'):
    metrics = [r[method] for r in runs]
    row = dict(method=method, calls=sum(x['calls'] for x in metrics),
               input_tokens=sum(x['input_tokens'] for x in metrics),
               output_tokens=sum(x['output_tokens'] for x in metrics),
               generate_sec=sum(x['generate_sec'] for x in metrics),
               format_matches=sum(x['format_match'] for x in metrics), n=len(metrics),
               cash_cost=None, peak_rss=None, energy=None)
    comparison.append(row)
    print('구성 비교:', row)
transitions = {key:0 for key in ('유지 정답','교정','퇴행','유지 실패')}
for r in runs:
    before, after = r['direct']['format_match'], r['refine']['format_match']
    key = '유지 정답' if before and after else '교정' if after else '퇴행' if before else '유지 실패'
    transitions[key] += 1
    print(r['case_id'], key, '초안:',r['stages'][0]['text'], '수정:',r['stages'][2]['text'])
assert sum(transitions.values()) == 3
print('전이:',transitions)


구성 비교: {'method': 'direct', 'calls': 3, 'input_tokens': 681, 'output_tokens': 50, 'generate_sec': 10.675539800000024, 'format_matches': 1, 'n': 3, 'cash_cost': None, 'peak_rss': None, 'energy': None}
구성 비교: {'method': 'refine', 'calls': 9, 'input_tokens': 2233, 'output_tokens': 194, 'generate_sec': 38.24404249999975, 'format_matches': 0, 'n': 3, 'cash_cost': None, 'peak_rss': None, 'energy': None}
cities 유지 실패 초안: 제주특별자치도의 두 행정시 이름은 제주시와 서귀포시이다. 수정: 제주특별자치도의 두 행정시 이름은 제주시와 서귀포시이다.
문서
count 퇴행 초안: 2 수정: 제주특별자치도의 행정시 수는 2개인 제주시와 서귀포시가
other 유지 실패 초안: 제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제 수정: 제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제
전이: {'유지 정답': 0, '교정': 0, '퇴행': 1, '유지 실패': 2}


## 6. 결과 저장, 한 장 기록, 확인 질문

작성 결과: 논문 한 장과 작은 재현 관찰

| 기록 항목 | 확인 근거와 관찰 |
|---|---|
| 서지·문제 | Aman Madaan 외, Self-Refine: Iterative Refinement with Self-Feedback, 2023, arXiv v2(2023-05-25). 첫 생성의 요구사항 미충족을 추가 학습 없이 개선하려는 방법. [원문 첫 페이지·§1](https://arxiv.org/html/2303.17651v2) |
| 논문 주장·근거 | 동일 모델의 직접 생성 대비 자기 피드백·반복 수정의 개선을 7개 생성 과업에서 평가. Table 1의 과업별 자동 지표·모델 평가와 §3.2의 별도 사람 선호 평가에 근거한 주장. [§3](https://arxiv.org/html/2303.17651v2#S3) |
| 방법 대응 | 원문 §2의 생성 -> 피드백 -> 수정 전달 구조를 제공 코드의 initial -> feedback -> refined로 한 회 실행. 원문은 few-shot과 중단 조건을 사용하며 이번 코드는 한 회 후 종료. [§2](https://arxiv.org/html/2303.17651v2#S2) |
| 실제 조건 | Qwen/Qwen3-0.6B-Base, revision da87bfb608c14b7cf20ba1ce41287e8de496c0cd, CPU float32, 2스레드, seed=0, greedy, 최대 24토큰. 한국어 위키백과 고정판 42464087의 실제 수집 문단과 교수 작성 3개 질문·기대 답 사용 |
| 작은 재현 근거 | 직접 형식 일치 1/3 -> 수정 0/3, 교정 0·퇴행 1·유지 실패 2·유지 정답 0. 생성 합 10.6755초 -> 38.2440초. 단계별 원 출력·토큰·시간은 코드 4절 출력과 JSON에 보존 |
| 저자 명시 한계 | 충분한 few-shot·지시 수행 능력 필요, 영어 중심 평가, 추가 생성 비용. [§6](https://arxiv.org/html/2303.17651v2#S6) |
| 자체 한계·차이 | 0.6B Base, 한국어 사실 질문 3건, 예시 없는 짧은 프롬프트, 24토큰 제한, 한 회 수정, 엄격한 첫 줄 형식 채점, 단일 고정 순서 측정. 원 논문의 모델·과업·반복·평가와 다른 조건이며 원 논문 성능 재현·반증 또는 SAIGE 구현은 아님 |
| 적용 판단(상정) | 추가 대기에도 이번 형식 교정 이득 없음. GPU 없는 공개 정보 초안 검토 시 직접 생성부터 사람이 확인하는 제한적 활용안. 자동 답변 확정·현장 운영 승인에 해당하지 않음 |


1. 같은 초안에서 피드백을 붙였을 때 답이 좋아졌는지 출력으로 확인
2. 수정 후 실패하면 피드백 오류·수정 반영 실패·출력 형식 오류를 구분
3. 호출 수 비율과 시간 비율이 다른 이유를 토큰 표로 설명
4. 동일 가중치 역할 분리가 독립 검토자의 근거가 되는지 설명

- 산출물: JSON과 위 한 장 기록, 실행 결과와 관찰 기록을 준비한 뒤 사용
- 이름·연락처 없이 학번 폴더만 사용, 원문 전체·API 키 제출 금지
- 실제 모델 출력은 공개 원문의 일부 문장을 인용할 수 있음. 원문 CC BY-SA 출처·기여자·이용조건 메타데이터를 결과와 함께 보존

□ 실제 실행 구성 비교

| 판단 항목 | 직접 생성 | 같은 모델 역할 분리·순차 수정 | 기록 근거 |
|---|---|---|---|
| 전용 GPU | 불필요 | 불필요 | device=cpu로 실제 실행 |
| 가중치 인스턴스 | 1 | 동일 1개 공유 | 같은 model 객체, 추가 모델 로드 없음 |
| 질문당 호출 / 3질문 합 | 1 / 3 | 3 / 9 | initial을 두 구성에서 공유 |
| 모델 로드 시간 | 4.2980초 | 공유, 중복 합산 없음 | conditions.load_sec, 생성 시간에서 제외 |
| 가중치 바이트 | 2,384,199,680 | 동일 2,384,199,680 | 596,049,920개 x float32 4바이트, 약 2.220 GiB |
| 전체 최대 RSS | 미측정 | 미측정 | 가중치 외 KV 캐시·중간 텐서·프로세스 포함 RAM과 구별 |
| 입력 토큰 합 | 681 | 2233 | 비교 구성의 모든 단계 합 |
| 출력 토큰 합 | 50 | 194 | 특수 토큰을 포함한 실제 생성 길이 |
| 생성 구간 시간 합 | 10.6755초 | 38.2440초 | model.generate 구간, 토큰화·디코딩 제외 |
| 첫 줄 형식 일치 | 1/3 | 0/3 | 문장 전체 의미 정확도와 다른 제공 지표 |
| 현금 비용·전력 | 미측정 | 미측정 | JSON null 유지, 추정 단가·전력값 없음 |
| 실패 확인 대상 | 근거·초안 | 근거·초안·피드백·수정 | 아래 사례별 검토 |
| 적용 판단(상정) | 질문당 생성 평균 3.5585초, 사람 검토 포함 시 총 대기는 별도 확인 필요 | 평균 12.7480초, 이번 교정 0건으로 추가 대기의 이점 미관찰 | 단회 로컬 CPU 값이며 기관 서비스 지연 기준 미확인 |

- 실제 비교를 위한 고유 생성은 9회. direct의 3회가 refine의 initial과 동일하므로 표의 3+9를 12회 실행으로 합산하지 않음. 별도 예열 1회는 비교 제외, JSON에도 비교 단계만 저장.
- 실행 호환성 보완: 없는 패키지만 설치하는 setup, 공개 수업 URL을 사용한 User-Agent, `do_sample=False`에 `use_model_defaults=False`를 명시하여 모델 기본값 재상속을 차단한 greedy 실행. 생성 조건에도 같은 값 기록. 제공 프롬프트·질문·단계·채점 함수 유지, 알고리즘·훈련 추가 없음.
- 실행 환경: torch 2.8.0+cpu, transformers 4.57.6, Python 3.10.0, AMD64. 진행 표시 위젯 미설치 경고와 torch_dtype 사용 중단 예정 경고가 있었으나 코드 7셀 오류 0. 실제 출력은 변경하지 않고 보존.

□ 확인 질문 답변

1. **같은 초안에서 피드백 후 개선 여부: 형식 기준으로 개선 없음.** cities는 초안과 수정 모두 두 이름을 포함하지만 이름만 나열한 답이 아니라 설명 문장이므로 실패. count는 초안 `2`가 통과했으나 수정은 `제주특별자치도의 행정시 수는 2개인 제주시와 서귀포시가`로 바뀌어 형식 퇴행. 숫자 자체가 2에서 오답 숫자로 바뀐 사례는 아님. other는 초안·수정 모두 `제주특별자치도에서 제주시 외의 행정시 이름은 제주특별자치도 제`로 끝나고 기대 이름 `서귀포시` 부재. 첫 줄 형식 점수의 1/3 -> 0/3을 의미 정확도의 전수 평가로 해석하지 않는 조건.

2. **피드백 오류, 반영 실패, 형식 실패의 분리.** 아래는 실제 출력의 수동 검토이며 독립 평가자 라벨은 아님.

| 사례 | 실제 피드백 관찰 | 수정 반영 판정 | 출력 실패의 구분 |
|---|---|---|---|
| cities | 맞는 두 이름의 문장을 반복하고 다음 줄에 `문서` 출력. 구체적인 수정 지점 없음 | 수정이 피드백과 동일, 제공된 유효 교정 지시가 없어 반영 능력만 분리 판정 불가 | 첫 줄의 설명 문장으로 형식 실패, 두 이름의 사실 자체는 근거와 부합 |
| count | 숫자 2를 포함한 불완전한 설명으로 확장. 숫자만 출력하라는 제약의 교정안 없음 | 수정이 피드백과 동일. 잘못된 교정 지시를 따랐다고 단정할 근거도 없음 | 초안 숫자 2는 통과, 수정은 숫자만인 답이 아니므로 형식 퇴행. 사실 오답 숫자 발생과 구별 |
| other | 질문을 반복하다 `제`에서 끝나며 서귀포시 제시 없음 | 초안·피드백·수정 동일, 유효한 수정 지시 없음 | 답 부재와 형식 실패 동시 관찰. 올바른 피드백을 주었을 때도 실패할지는 미측정 |

   - 피드백과 수정의 모든 출력이 24토큰 상한에 도달한 기록. 미완결 문장과 관련된 조건이나, 제한을 늘린 대조 실험이 없으므로 상한만을 실패의 유일한 원인으로 확정하지 않음.

3. **호출 비율 3.0배와 시간 비율 3.5824배의 차이.** 입력은 2233/681=3.2790배, 출력은 194/50=3.88배. 피드백·수정 단계에는 초안·피드백이 누적되어 입력 길이가 늘고, count의 직접 생성은 3토큰인데 피드백과 수정은 각각 24토큰이므로 호출별 계산량도 다름. 아래 생성 시간은 각각 실제 측정값이며 토큰 수만의 정확한 선형 함수나 호출 비용의 고정 배수로 간주하지 않음.

| 사례 | initial 입력/출력 토큰 | feedback 입력/출력 토큰 | refined 입력/출력 토큰 | 직접 / 세 단계 생성 합(초) |
|---|---|---|---|---|
| cities | 226/23 | 245/24 | 281/24 | 4.5755 / 13.8415 |
| count | 225/3 | 225/24 | 261/24 | 1.6332 / 10.4591 |
| other | 230/24 | 252/24 | 288/24 | 4.4668 / 13.9434 |

   - 고정 순서의 한 번 측정으로 반복 분산·신뢰구간·순서 효과 미측정. 다운로드·설치·모델 로드·예열은 생성 합에서 제외.

4. **동일 가중치의 역할 구분은 독립 검토자의 증거가 아님.** 세 역할이 같은 model 객체와 공개 근거를 사용하고, 수정 프롬프트는 초안과 피드백도 이어받는 구조. 이번 세 사례에서 피드백과 수정 문자열이 모두 같다는 관찰은 정보 의존을 보여 주지만 오류 상관의 확률적 추정까지 제공하지 않음. 외부 평가자·별도 모델·독립 검증 자료와의 비교가 없으므로 세 역할의 동의나 반복을 독립 전문가 합의로 보지 않는 판단.

- 최종 산출물: 실제 단계 출력·조건·합계·전이를 담은 `week-13-observations.json`과 위 논문/관찰/구성 비교 기록. 고정판 원문은 공개 페이지에서 실제 수집, 질문·정답 목록과 형식 규칙은 제공 예제, 시간·토큰·출력은 이번 실행의 실측값으로 구분. 원문 전체를 JSON에 저장하지 않았으며 공개 출처·판·기여자·CC BY-SA 4.0 정보 보존. 2026-11-28 재실행 결과를 GitHub 과제 폴더에 제출하는 관찰 기록. 생성형 AI의 실행 및 정리 보조 사용.


In [7]:
report = dict(kind='Self-Refine one-cycle structural reproduction, not paper benchmark reproduction',
              source_metadata=source_metadata, conditions=conditions,
              runs=runs, comparison=comparison, transitions=transitions,
              limitations=['3 factual questions','base model','strict first-line format match',
                           'single timing run','no independent models','no SAIGE implementation'])
path = Path('week-13-observations.json')
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
assert 'context' not in report and 'evidence' not in report
print('저장:', path)


저장: week-13-observations.json
